#common

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *
import pyspark.sql.functions as F
from pyspark.sql.utils import AnalysisException
from pyspark.sql.types import StructType, BooleanType

import pandas as pd
from datetime import datetime, timedelta, timezone

In [0]:
display( 
        spark.read.option("header", "true")
        .csv(f"/Volumes/catalog_southeastasia_mdm_share_prod/share_mdm_config/mdm_config_files/prod_cdp_mdm_history_load/prod_consumer_loading_delta_config_20260714.csv")
)


In [0]:
# 定义 UTC+8 时区
TZ_UTC8 = timezone(timedelta(hours=8))

def print_log(message, level="INFO"):
    """打印带 UTC+8 时间戳的日志"""
    timestamp = datetime.now(TZ_UTC8).strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{timestamp}] [{level}] {message}")

# ==============================================
# 【新增】Schema 表结构对比函数：打印字段/类型差异
# ==============================================
def compare_schemas(source_df, target_df, source_name="Source表", target_name="Target表"):
    """
    对比两个DataFrame的Schema结构，输出差异：
    1. Source有、Target没有的字段
    2. Target有、Source没有的字段
    3. 字段名相同、数据类型不同的字段
    """
    # 提取字段名+类型 转字典
    source_schema = {f.name.upper(): str(f.dataType).upper() for f in source_df.schema.fields}
    target_schema = {f.name.upper(): str(f.dataType).upper() for f in target_df.schema.fields}
    
    source_fields = set(source_schema.keys())
    target_fields = set(target_schema.keys())
    
    # 1. 独有字段
    source_only = source_fields - target_fields
    target_only = target_fields - source_fields
    
    # 2. 同名字段类型不同
    type_mismatch = []
    common_fields = source_fields & target_fields
    for field in common_fields:
        s_type = source_schema[field]
        t_type = target_schema[field]
        if s_type != t_type:
            type_mismatch.append(f"{field} | Source类型:{s_type} | Target类型:{t_type}")
    
    # 格式化打印差异
    print("-" * 10)
    print(f"📊 【表结构差异对比】{source_name} VS {target_name}")
    print("-" * 10)
    print(f"🔹 {source_name} 独有字段 ({len(source_only)}个): {sorted(source_only) if source_only else '无'}")
    print(f"🔹 {target_name} 独有字段 ({len(target_only)}个): {sorted(target_only) if target_only else '无'}")
    print(f"🔹 字段类型不一致 ({len(type_mismatch)}个):")
    if type_mismatch:
        for idx, msg in enumerate(type_mismatch, 1):
            print(f"   {idx}. {msg}")
    else:
        print("   无")


In [0]:
def align_schema(source_df: DataFrame, target_df: DataFrame) -> DataFrame:
    """
    自动将 source_df 的字段类型转换为与 target_df 完全一致的类型
    字段名必须匹配，自动按 target_df 字段顺序重排，自动类型强转
    
    参数：
        source_df: 原始数据源 DataFrame
        target_df: 目标表结构 DataFrame（用于获取目标 schema）
    
    返回：
        类型对齐后的新 DataFrame
    """

    # source_fields = [f.upper() for f in source_df.columns]
    # source_schema = source_df.schema
    # # 获取目标表的所有字段名 + 类型
    # target_fields = [f.upper() for f in target_df.columns]
    # target_schema = target_df.schema


    # 提取字段名+类型 转字典
    source_schema = {f.name.upper(): f.dataType for f in source_df.schema.fields}
    source_fields = set(source_schema.keys())

    target_schema = {f.name.upper(): f.dataType for f in target_df.schema.fields}
    target_fields = set(target_schema.keys())


    # 只保留 source_df 和 target_df 共有的字段
    common_fields = [field for field in target_fields if field in source_fields]
    # print(f"common_fields: {common_fields}")

    # 为每个共有字段执行【自动类型转换】，并按 target 顺序排列
    casted_cols = []

    for field in common_fields:
        s_type = source_schema[field]
        t_type = target_schema[field]

        if s_type != t_type:
            casted_cols.append(
                F.col(field).cast(t_type).alias(field)
                )
        else:
            casted_cols.append(F.col(field))

    # 生成最终对齐后的 DataFrame
    result_df = source_df.select(*casted_cols)
    # result_df.printSchema()

    return result_df

In [0]:
def get_target_table_name(tartget_db: str, tartget_tb: str) -> str:
    return f"{tartget_db}.{tartget_tb}"

In [0]:
# MySQL 连接信息
# MYSQL_HOST     = dbutils.secrets.get('sr-env', 'APAC_MDM_MYSQL_HOST')
# MYSQL_USER     = dbutils.secrets.get('sr-env', 'APAC_MDM_MYSQL_USER')
# MYSQL_PASSWORD = dbutils.secrets.get('sr-env', 'APAC_MDM_MYSQL_PSWD')
MYSQL_DRIVER   = "com.mysql.cj.jdbc.Driver"

MYSQL_HOST = "mysqlflex-ap-southeastasia-prod-cepa-talend-02.mysql.database.azure.com"
MYSQL_USER = "talend_mdm_read_only@mysql-ap-southeastasia-prod-cepa-talend-02"
MYSQL_PASSWORD = "Q4&4c9JpMSb5A=xa"

# market -> MySQL 库名映射（库名 = market小写 + "_elcconsumermdm"）
MARKET_DB_MAP = {
    "AUS": "aus_elcconsumermdm",
    "HKG": "hkg_elcconsumermdm",
    "IDN": "idn_elcconsumermdm",
    "JPN": "jpn_elcconsumermdm",
    "KOR": "kor_elcconsumermdm",
    "MYS": "mys_elcconsumermdm",
    "NZL": "nzl_elcconsumermdm",
    "PHL": "phl_elcconsumermdm",
    "SGP": "sgp_elcconsumermdm",
    "THA": "tha_elcconsumermdm",
    "TWN": "twn_elcconsumermdm",
    "VNM": "vnm_elcconsumermdm",
}
MYSQL_TABLE  = "sconsumer"
PK_COLS      = ["scon_mrkt_code", "scon_brnd_code", "scon_srcs_code", "scon_consumerid"]

def build_jdbc_url(database: str) -> str:
    """构建 MySQL JDBC URL"""
    return (
        f"jdbc:mysql://{MYSQL_HOST}:3306/{database}"
        f"?serverTimezone=UTC&useUnicode=true&characterEncoding=UTF-8"
        f"&zeroDateTimeBehavior=CONVERT_TO_NULL&useSSL=true&enabledTLSProtocols=TLSv1.2"
    )


def query_mysql_pk_exists(database: str, table: str, pk_rows: list, pk_cols: list):
    """
    将 a 表收集到的主键列表作为 WHERE 条件下推到 MySQL，
    只返回在 MySQL 中存在的主键行，避免全表扫描大表。

    pk_rows: list of Row，每个 Row 包含 pk_cols 对应的字段值
    返回: DataFrame，含 pk_cols 列（MySQL 中存在的主键集合）
    """
    if not pk_rows:
        return spark.createDataFrame([], schema="scon_mrkt_code STRING, scon_brnd_code STRING, scon_srcs_code STRING, scon_consumerid STRING")

    # 构建 WHERE 条件: (col1='v1' AND col2='v2' AND ...) OR ...
    # 字符串值需要转义单引号，防止 SQL 注入
    def escape(v):
        return str(v).replace("'", "''")

    row_conditions = []
    for row in pk_rows:
        col_conds = [f"{col} = '{escape(row[col])}'" for col in pk_cols]
        row_conditions.append("(" + " AND ".join(col_conds) + ")")

    where_clause = " OR ".join(row_conditions)
    pk_select    = ", ".join(pk_cols)
    query = f"(SELECT {pk_select} FROM {table} WHERE {where_clause}) AS pk_subquery"

    jdbc_url   = build_jdbc_url(database)
    jdbc_props = {
        "user":     MYSQL_USER,
        "password": MYSQL_PASSWORD,
        "driver":   MYSQL_DRIVER,
    }
    return spark.read.jdbc(url=jdbc_url, table=query, properties=jdbc_props)

In [0]:
# 增加 is_retain_for_history 字段
def check_retain(df_a):
    # 2. scon_mrkt_code 不在12个 market 中的数据，直接标记 is_retain_for_history = false
    known_markets = list(MARKET_DB_MAP.keys())

    df_unknown_market = (
        df_a
        .filter(~F.col("scon_mrkt_code").isin(known_markets))
        .withColumn("is_retain_for_history", F.lit(False).cast(BooleanType()))
    )

    # 只对已知 market 的数据做 MySQL 主键校验
    df_a = df_a.filter(F.col("scon_mrkt_code").isin(known_markets))

    # 3. 按 market 逐个处理，收集结果后 union
    result_dfs = []

    for market, database in MARKET_DB_MAP.items():

        # 过滤出当前 market 的数据
        df_market = df_a.filter(F.col("scon_mrkt_code") == market)

        # 将 a 表该 market 的主键收集到 driver（行数少，开销可忽略）
        pk_rows = df_market.select(PK_COLS).distinct().collect()
        # print(f"[{market}] a 表主键行数: {len(pk_rows)}，查询 MySQL: {database}.{MYSQL_TABLE}")

        if len(pk_rows) == 0:
            # 无数据时直接标记 is_retain_for_history = true (实际不会执行)
            df_result = df_market.withColumn("is_retain_for_history", F.lit(True).cast(BooleanType()))
        else:
            # 把主键列表下推到 MySQL，只返回 MySQL 中存在的那些主键
            df_mysql_pk = query_mysql_pk_exists(database, MYSQL_TABLE, pk_rows, PK_COLS)

            # left join: 若 MySQL 侧主键存在则标记 _exists_flag = true
            df_joined = df_market.join(
                df_mysql_pk.withColumn("_exists_flag", F.lit(True)),
                on=PK_COLS,
                how="left"
            )

            # 主键存在 → is_retain_for_history = false; 不存在（null）→ true
            df_result = df_joined.withColumn(
                "is_retain_for_history",
                F.when(F.col("_exists_flag").isNotNull(), False).otherwise(True).cast(BooleanType())
            ).drop("_exists_flag")

        result_dfs.append(df_result)
        # print(f"[{market}] done")


    # 4. 合并所有 market 结果 + 未知 market 结果
    df_final = result_dfs[0]
    for df in result_dfs[1:]:
        df_final = df_final.unionByName(df)

    df_final = df_final.unionByName(df_unknown_market)

    return df_final


In [0]:
# import logging
# import os
# from datetime import datetime

# class DatabricksLogger:
#     def __init__(self, log_name, log_dir="/dbfs/tmp/logs"):
#         # 确保日志目录存在
#         # os.makedirs(log_dir, exist_ok=True)
        
#         log_file = f"{log_dir}/{log_name}_{datetime.now().strftime('%Y%m%d')}.log"
        
#         self.logger = logging.getLogger(log_name)
#         self.logger.setLevel(logging.DEBUG)
        
#         # 文件处理器
#         file_handler = logging.FileHandler(log_file)
#         file_handler.setLevel(logging.DEBUG)
        
#         # 控制台处理器（显示在笔记本单元格）
#         console_handler = logging.StreamHandler()
#         console_handler.setLevel(logging.INFO)
        
#         # 格式化
#         formatter = logging.Formatter(
#             '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
#         )
#         file_handler.setFormatter(formatter)
#         console_handler.setFormatter(formatter)
        
#         self.logger.addHandler(file_handler)
#         self.logger.addHandler(console_handler)
    
#     def get_logger(self):
#         return self.logger

# # 使用
# logger = DatabricksLogger(log_name ="my_app", log_dir ="/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417").get_logger()

# logger.info("数据处理开始")
# # logger.error("发生错误", exc_info=True)

In [0]:
# t_df = spark.createDataFrame([("aa",1)], ["name","age"])

# l_st = t_df.toPandas().to_string(index=False)
# # print(l_st)

# logger.info(l_st)

# # l_time = datetime.now().strftime("%Y-%m-%d_%H:%M:%S")
# # l_path = f"/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/{l_time}.log"
# # print(l_path)

# # print_in_log_file(l_path,l_st)

#main

In [0]:
import pyspark.sql.functions as F

sub_table_list = [
    "sconsumeraddress",
    "sconsumerauxiliaryattribute",
    "sconsumergroup",
    "sconsumercrossbrandoptin",
    "sconsumercustomattr",
    "sconsumermedia",
    "sconsumerhairconcerns",
    "sconsumerhairtype",
    "sconsumerhobby",
    "sconsumermakeupconcerns",
    "sconsumernotes",
    "sconsumeroptin",
    "sconsumerphone",
    "sconsumerprogram",
    "sconsumerremark",
    "sconsumerskinconcerns",
    "sconsumerterms"
]

current_timesamp_str = "20260721090000"

data_loading_config =spark.read.option("header", "true").csv(f"/Volumes/catalog_southeastasia_mdm_share_prod/share_mdm_config/mdm_config_files/prod_cdp_mdm_history_load/prod_consumer_loading_delta_config_20260714.csv")

# 读取配置表，只处理激活状态的数据 
config_df = data_loading_config.filter("is_loading_delta_active = true  ").orderBy(col("index").cast("int").asc()) \
    .where(" market not in ('SGP', 'TWN')  ")
display(config_df)

# 额外读取ssourceconsumer表配置, 用于补充主表字段
ssourceconsumer_config_df = data_loading_config.filter("source_table = 'ssourceconsumer' ")

sconsumer_config_df = data_loading_config.filter("source_table = 'sconsumer' ")

In [0]:


# 遍历每一条配置（逐表同步）
for row in config_df.collect():
    # 提取配置字段
    market = row["market"].upper()
    source_db = row["source_database"]
    source_tb = row["source_table"]
    tartget_db = row["tartget_database"]
    tartget_tb = row["target_table"]

    target_blob_path = row["target_path"]
    primary_key = row["primary_key"] 
    # pk_type = row["pk_type"]  
    Markt_Column = row["Markt_Column"]
    foreign_key = row["foreign_key"]

    index = row["index"]

    # 目标表名
    target_table = get_target_table_name(tartget_db, tartget_tb)

    print_log("="*20)
    print_log(index)
    print_log(f"market: {market}")
    print_log(f"=== 开始同步：{source_db}.{source_tb} -> {target_table} | 主键：{primary_key},{Markt_Column},{foreign_key} ===")


    # step01: 数据生成
    print_log(f"source table path: {target_blob_path}")
    source_df = spark.read.format("delta").load(target_blob_path)

    if source_tb == 'sconsumer':
        print_log("source_df: ")
        source_df.groupBy(Markt_Column).count().show()

        # 1. 补充字段
        ssourceconsumer_path = ssourceconsumer_config_df.filter(f"market = '{market}' ").head()["target_path"]
        print_log(f"补充字段 ssourceconsumer_path: {ssourceconsumer_path}")
        ssourceconsumer_df =spark.read.format("delta").load(ssourceconsumer_path)

        # 2. 字段修改
        update_df = (source_df.alias("sd").join(ssourceconsumer_df.alias("ssd"), F.col("sd.scon_srcc_id") == F.col("ssd.srcc_id"), "left") \
            .select(
                F.col("sd.*"),
                F.col('ssd.SRCC_MASTERCONSUMERID').alias('SCON_MASTERCONSUMERID') ,
                F.col('ssd.SRCC_SOURCESYSTEMCODE').alias('SCON_SOURCESYSTEMCODE') ,
                F.col('ssd.SRCC_UNIVERSALKEY').alias('SCON_UNIVERSALKEY')
            )
            .withColumn("task_id", F.concat(F.lit("init-"), F.lit(current_timesamp_str)))
            .withColumn("batch_id", F.concat(F.lit("init-"), F.lit(current_timesamp_str)))
        )
        

        # 3. 排除数据
        correct_df = update_df.filter(F.coalesce(F.col(Markt_Column), F.lit("unknow")) == market) \
            .withColumn("scon_id", F.concat(F.col(Markt_Column), F.lit("-"),  F.format_string('%09d', F.col("scon_id"))))  # scon_id 增加market和补0
        print_log("correct_df: ")
        correct_df.groupBy(Markt_Column).count().show()

        error_df = update_df.filter(F.coalesce(F.col(Markt_Column), F.lit("unknow")) != market)
        print_log("error_df: ")
        error_df.groupBy(Markt_Column).count().show()

        # 3.1 保存sconsumer主表的交叉数据
        check_retain_df = check_retain(error_df)
        check_retain_df.cache()

        print_log("check_retain_df: ")
        check_retain_df.groupBy(Markt_Column, "is_retain_for_history").count().show()

        cross_market_data_path = target_blob_path + "_cross_market"
        print_log(f"cross_market_data_path: {cross_market_data_path}")
        check_retain_df.write.format("delta").mode("overwrite").save(cross_market_data_path)

        pass_exclued_df = check_retain_df \
            .filter(F.col("is_retain_for_history") == True) \
            .withColumn("scon_id", F.concat(F.col(Markt_Column), F.lit(f"-{source_db}-"), F.format_string('%09d', F.col("scon_id")))) \
            .drop("is_retain_for_history")

        print_log("pass_exclued_df: ")
        pass_exclued_df.groupBy(Markt_Column).count().show()
        
        print_log("result_df: ")
        result_df = correct_df.unionByName(pass_exclued_df)
        result_df.groupBy(Markt_Column).count().show()


        # 4. 数据写入
        target_df = spark.table(target_table).limit(1)
        # 字段类型处理
        compare_schemas(result_df, target_df, source_name="Source(Blob)表", target_name="Target(Databricks)表")
        write_df = align_schema(result_df,target_df)

        (write_df.write.format("delta").mode("append").saveAsTable(target_table))

        check_retain_df.unpersist()
    
    elif source_tb in sub_table_list:
        print_log("source_df: ")
        print_log(source_df.count())

        # 1.1 补充字段 market 字段
        sconsumer_path = sconsumer_config_df.filter(f"market = '{market}' ").head()["target_path"]
        print_log(f"补充market字段 sconsumer_path: {sconsumer_path}")
        sconsumer_df = spark.read.format("delta").load(sconsumer_path)

        # 1.2 补充字段 is_retain_for_history 字段
        cross_market_data_path = sconsumer_path + "_cross_market"
        print_log(f"补充 is_retain_for_history 字段 cross_market_data_path: {cross_market_data_path}")
        cross_market_df = spark.read.format("delta").load(cross_market_data_path)

        # 2. 字段修改
        update_df = (source_df.alias("sd")
            .join(sconsumer_df.alias("scd"), F.col(f"sd.{foreign_key}") == F.col("scd.scon_id"), "left") \
            .join(cross_market_df.alias("cmd"), F.col(f"sd.{foreign_key}") == F.col("cmd.scon_id"), "left") \
            .select(
                F.col("sd.*"),
                F.coalesce(F.col("scd.scon_mrkt_code"), F.lit(market)).alias(Markt_Column) ,
                F.col("cmd.is_retain_for_history").alias("is_retain_for_history") ,
            )
            .withColumn("task_id", F.concat(F.lit("init-"), F.lit(current_timesamp_str)))
            .withColumn("batch_id", F.concat(F.lit("init-"), F.lit(current_timesamp_str)))
            .withColumn(primary_key, 
                F.when(F.col("is_retain_for_history") == True,  F.concat(F.col(Markt_Column), F.lit(f"-{source_db}-"), F.format_string('%09d', F.col(primary_key).cast("int"))))
                .otherwise(F.concat(F.col(Markt_Column), F.lit("-"),  F.format_string('%09d', F.col(primary_key).cast("int"))))
            )
            .withColumn(foreign_key, 
                F.when(F.col("is_retain_for_history") == True,  F.concat(F.col(Markt_Column), F.lit(f"-{source_db}-"), F.format_string('%09d', F.col(foreign_key).cast("int"))))
                .otherwise(F.concat(F.col(Markt_Column), F.lit("-"),  F.format_string('%09d', F.col(foreign_key).cast("int") ))   )
            )
        )

        update_df.groupBy(Markt_Column, "is_retain_for_history").count().show()

        # 3. 排除数据
        correct_df = update_df.filter((F.col("is_retain_for_history").isNull()) | (F.col("is_retain_for_history") == True))
        print_log("correct_df: ")
        correct_df.groupBy(Markt_Column, "is_retain_for_history").count().show()

        error_df = update_df.filter(F.col("is_retain_for_history") == False)
        print_log("error_df: ")
        error_df.groupBy(Markt_Column, "is_retain_for_history").count().show()

        # 4. 数据写入
        target_df = spark.table(target_table).limit(1)
        # 字段类型处理
        compare_schemas(correct_df, target_df, source_name="Source(Blob)表", target_name="Target(Databricks)表")
        write_df = align_schema(correct_df,target_df)

        (write_df.write.format("delta").mode("append").saveAsTable(target_table))

    elif source_tb in ["cbrl", "cbrlpublic", "cbrldrjart", "cbrl_inc", "cbrlpublic_inc"]:
        # 增加task_id
        update_df = source_df.withColumn("task_id", F.concat(F.lit("init-"), F.lit(current_timesamp_str)))
        
        # 数据写入
        target_df = spark.table(target_table).limit(1)
        # 字段类型处理
        compare_schemas(update_df, target_df, source_name="Source(Blob)表", target_name="Target(Databricks)表")
        write_df = align_schema(update_df,target_df)

        (write_df.write.format("delta").mode("append").saveAsTable(target_table))        

    elif source_tb in ["consumertransactionlist_linegift", "consumertransactionlist_rakuten"]:
        # 字段更名
        # 增加task_id
        update_df = (source_df
            .select(
                F.col("scon_creation_dt").alias("creation_dt"),
                F.col("scon_srcc_action").alias("tran_action"),
                F.col("scon_brnd_code").alias("tran_brnd_code"),
                F.col("scon_id").alias("tran_id"),
                F.col("consumermdmkey").alias("tran_mapping_consumer_id"),
                F.col("scon_mrkt_code").alias("tran_mrkt_code"),
                F.col("scon_consumerid").alias("tran_order_id"),
                F.col("scon_srcc_id").alias("tran_srcc_id"),
                F.col("scon_srcs_code").alias("tran_srcs_code"),
                F.col("scon_update_dt").alias("update_dt")
            )
            .withColumn("task_id", F.concat(F.lit("init-"), F.lit(current_timesamp_str)))
        )
        
        # 数据写入
        target_df = spark.table(target_table).limit(1)
        # 字段类型处理
        compare_schemas(update_df, target_df, source_name="Source(Blob)表", target_name="Target(Databricks)表")
        write_df = align_schema(update_df,target_df)

        (write_df.write.format("delta").mode("append").saveAsTable(target_table))   
    
    elif source_tb == "line_unbind_exclusion":
        # 增加 scme_market_code, 固定值: JPN
        update_df = source_df \
            .select(
                F.lit(None).alias("scme_srcc_id"),
                F.lit("JPN").alias("scme_market_code"),
                F.col("scme_emdt_code"),
                F.col("scme_sourcetimestamp"),
                F.col("scme_address"),
                F.col("scme_validitycode"),
                F.lit(None).alias("scme_primary_flag"),
                F.lit(None).alias("scme_appid"),
                F.lit(None).alias("scme_contactoptinflag"),
                F.lit(None).alias("scme_referenceemediatypecode"),
                F.lit(None).alias("scme_referenceemediaaddress"),
                F.lit(None).alias("scme_quality_code"),
                F.lit(None).alias("scme_quality_desc"),
                F.col("scme_creation_dt"),
                F.lit(None).alias("scme_creation_uid"),
                F.col("scme_update_dt"),
                F.lit(None).alias("scme_update_uid"),
                F.lit(None).alias("scme_update_flag")
            )
            
        
        # 数据写入
        target_df = spark.table(target_table).limit(1)
        # 字段类型处理
        compare_schemas(update_df, target_df, source_name="Source(Blob)表", target_name="Target(Databricks)表")
        write_df = align_schema(update_df,target_df)

        (write_df.write.format("delta").mode("append").saveAsTable(target_table))        

    elif source_tb == "sconsumermedia_line_unbind_records":
        # 增加 scme_market_code, 固定值: THA
        update_df = source_df.withColumn("scme_market_code", F.lit("THA"))
        
        # 数据写入
        target_df = spark.table(target_table).limit(1)
        # 字段类型处理
        compare_schemas(update_df, target_df, source_name="Source(Blob)表", target_name="Target(Databricks)表")
        write_df = align_schema(update_df,target_df)

        (write_df.write.format("delta").mode("append").saveAsTable(target_table)) 

    # elif source_tb == "sconsumermedia_unbind_history":
    #     # 增加 scme_market_code, 固定值: THA
    #     update_df = source_df.withColumn("scme_market_code", F.lit("THA"))
        
    #     # 数据写入
    #     target_df = spark.table(target_table).limit(1)
    #     # 字段类型处理
    #     compare_schemas(update_df, target_df, source_name="Source(Blob)表", target_name="Target(Databricks)表")
    #     write_df = align_schema(update_df,target_df)

    #     (write_df.write.format("delta").mode("append").saveAsTable(target_table)) 

    else:
        print_log("!"*30)
        print_log(f"{source_db}.{source_tb} 未配置处理逻辑")




print_log("=== 所有配置表任务执行完毕 ===")



# 1. 补充字段
# 1.1 子表补充 market 字段
# 1.2 sconsumer 补充 SCON_MASTERCONSUMERID, SCON_SOURCESYSTEMCODE, SCON_UNIVERSALKEY (从 ssourceconsumer中获取 )

# 2. 字段修改 增加task_id:  F.concat(F.lit("init-"), F.lit(current_timesamp_str))

# 3. 主表 market交叉数据处理
# 3.1 查询mysql 数据, 确认交叉数据是否保留: (必须查询mysql, 因为是分market拉取历史数据)
#   添加 is_retain_for_history 字段: true -> 需要写入 databricks, false -> 不需要写入. 
# 3.2 修改保留的交叉数据主键 scon_id -> F.concat(F.lit(f"{source_db}-"), F.col("scon_id"))
# 3.3 交叉数据留存blob

# 4. 子表 market交叉数据处理
# 4.1 查询留存交叉数据判断是否子表数据写入databricks
#   is_retain_for_history 为 false 则过滤
#   is_retain_for_history 为 true , 修改主表外键 sxxx_scon_id -> F.concat(F.lit(f"{source_db}-"), F.col("sxxx_scon_id")), 
#                                 修改主键 sxxx_id -> F.concat(F.lit(f"{source_db}-"), F.col("sxxx_id"))

#query

In [0]:
%sql

select
  SCON_MRKT_CODE, count(*)
from
  catalog_southeastasia_mdm_silver_uat.history_data_loading.t_master_consumer
group by
  SCON_MRKT_CODE

In [0]:
%sql

select
  SCON_MRKT_CODE, SCON_ID, count(*)
from
  catalog_southeastasia_mdm_silver_uat.history_data_loading.t_master_consumer
group by
  SCON_MRKT_CODE, SCON_ID
having
  count(*) > 1

In [0]:
%sql

select
 *
from
  catalog_southeastasia_mdm_silver_uat.history_data_loading.t_master_auxiliary_attribute


In [0]:
%sql

select
  SCAA_MRKT_CODE,count(*)
from
  catalog_southeastasia_mdm_silver_uat.history_data_loading.t_master_auxiliary_attribute
group by
  SCAA_MRKT_CODE

In [0]:
%sql

select
  *
from
  catalog_southeastasia_mdm_silver_uat.history_data_loading.t_master_auxiliary_attribute
where
  SCAA_MRKT_CODE = 'HKG'

In [0]:
%sql

select
  *
from
  catalog_southeastasia_mdm_silver_uat.history_data_loading.t_master_consumer
where
  SCON_ID in ('aus_elcconsumermdm-41023792', 'aus_elcconsumermdm-41023836')

In [0]:
%sql

select
  SCAA_MRKT_CODE,count(*)
from
  catalog_southeastasia_mdm_silver_uat.history_data_loading.t_master_auxiliary_attribute
group by
  SCAA_MRKT_CODE, SCAA_ID
having
  count(*) >1 

In [0]:
# 示例：编译时认为是两个大表，但过滤后小表变小
df1 = spark.range(1000000).filter("id < 100")  # 实际只有100行
df2 = spark.range(1000)

# 无 AQE：可能使用 SortMergeJoin（有 shuffle）
# 有 AQE：检测到 df1 很小，自动切换为 BroadcastHashJoin
result = df1.join(df2, "id")
result.explain()

#数据前置探查

In [0]:
%sql

select
  market_source, database_name, scon_mrkt_code, count(*), case when market_source = scon_mrkt_code then true else false end as is_sense
from

(
select 'AUS' as market_source, 'aus_elcconsumermdm' as database_name, scon_mrkt_code from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/AUS/aus_elcconsumermdm/sconsumer` union all
select 'HKG' as market_source, 'hkg_elcconsumermdm' as database_name, scon_mrkt_code from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/HKG/hkg_elcconsumermdm/sconsumer` union all
select 'IDN' as market_source, 'idn_elcconsumermdm' as database_name, scon_mrkt_code from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/IDN/idn_elcconsumermdm/sconsumer` union all  
select 'JPN' as market_source, 'jpn_elcconsumermdm' as database_name, scon_mrkt_code from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/JPN/jpn_elcconsumermdm/sconsumer` union all
select 'KOR' as market_source, 'kor_elcconsumermdm' as database_name, scon_mrkt_code from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/KOR/kor_elcconsumermdm/sconsumer` union all
select 'MYS' as market_source, 'mys_elcconsumermdm' as database_name, scon_mrkt_code from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/MYS/mys_elcconsumermdm/sconsumer` union all
select 'NZL' as market_source, 'nzl_elcconsumermdm' as database_name, scon_mrkt_code from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/NZL/nzl_elcconsumermdm/sconsumer` union all
select 'PHL' as market_source, 'phl_elcconsumermdm' as database_name, scon_mrkt_code from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/PHL/phl_elcconsumermdm/sconsumer` union all
select 'SGP' as market_source, 'sgp_elcconsumermdm' as database_name, scon_mrkt_code from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/SGP/sgp_elcconsumermdm/sconsumer` union all
select 'THA' as market_source, 'tha_elcconsumermdm' as database_name, scon_mrkt_code from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/THA/tha_elcconsumermdm/sconsumer` union all
select 'TWN' as market_source, 'twn_elcconsumermdm' as database_name, scon_mrkt_code from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/TWN/twn_elcconsumermdm/sconsumer` union all
select 'VNM' as market_source, 'vnm_elcconsumermdm' as database_name, scon_mrkt_code from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/VNM/vnm_elcconsumermdm/sconsumer` 
)

group by
  all
order by
  all


In [0]:
%sql

select  
  *
from
  delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/AUS/aus_elcconsumermdm/sconsumer`
where
  scon_mrkt_code != 'AUS'

In [0]:
hkg_elcconsumermdm
scon_mrkt_code = 'HKG' and scon_brnd_code = '01' and  scon_srcs_code = 'undhkgundftsund' and scon_consumerid = 'U2AUSJJR10002'

kor_elcconsumermdm
scon_mrkt_code = 'KOR' and scon_brnd_code = '01' and  scon_srcs_code = '50kor10prlund' and scon_consumerid = '1001425822'

# 数据检验

In [0]:
%sql

select
  *
from
  catalog_southeastasia_mdm_silver_uat.history_data_loading.t_master_consumer

In [0]:
%sql

 select  
  SCON_MRKT_CODE, SCON_BRND_CODE, SCON_SRCS_CODE, SCON_CONSUMERID, count(*), collect_list(SCON_ID)
from 
  catalog_southeastasia_mdm_silver_uat.history_data_loading.t_master_consumer
group by
  SCON_MRKT_CODE, SCON_BRND_CODE, SCON_SRCS_CODE, SCON_CONSUMERID
having
    count(*) >1



In [0]:
%sql

 select  
  SCON_MRKT_CODE, count(*)
from 
  catalog_southeastasia_mdm_silver_uat.history_data_loading.t_master_consumer
group by
  SCON_MRKT_CODE 

In [0]:
%sql

select
  SCON_MRKT_CODE, cross_market, count(*)
from
  (
    select 
      *, case when contains(SCON_ID, '-') then split_part(SCON_ID, '-', 1) else null end as cross_market 
    from 
      catalog_southeastasia_mdm_silver_uat.history_data_loading.t_master_consumer
  )
group by
  SCON_MRKT_CODE, cross_market
order by
  SCON_MRKT_CODE, cross_market

In [0]:
dbutils.widgets.text("order_cols", "scon_id,scon_srcc_id,scon_srcc_action,consumermdmkey,scon_srcs_code,scon_sourcetimestamp,scon_mrkt_code,scon_aff_code,scon_dvsn_code,scon_brnd_code,scon_consumerid,scon_salutation,scon_englishfirstname,scon_englishmiddlename,scon_englishlastname,scon_englishfullname,scon_localfirstname,scon_localmiddlename,scon_locallastname,scon_localfullname,scon_localfirstname2,scon_localmiddlename2,scon_locallastname2,scon_localfullname2,scon_gndr_code,scon_birthday,scon_birthmonth,scon_birthyear,scon_identitynum,scon_passportnum,scon_socialsecuritynum,scon_clas_code,scon_reg_dt,scon_registration_toch_code,scon_registration_prsn_code,scon_preferred_toch_code,scon_assigned_prsn_code,scon_wlng_code,scon_slng_code,scon_cntr_isoalpha3code,scon_ethn_code,scon_sknt_code,scon_hairt_code,scon_cvls_code,scon_company,scon_department,scon_jobtitle,scon_yearlyincome,scon_donotcontact_flag,scon_curr_code,scon_agefrom,scon_ageto,scon_nationality,scon_channel,scon_preferred_comm_channel,scon_anniversary_dt,scon_commercial_flag,scon_emailreceipt_flag,scon_prospect_flag,scon_active_flag,scon_hrrequesttimestamp,scon_status,scon_creation_dt,scon_creation_uid,scon_update_dt,scon_update_uid,scon_cbr_flag,scon_firstpurchasedate,scon_delete_flag,scon_englishname_quality_code,scon_localname_quality_code,scon_localname2_quality_code")

# SCON_MASTERCONSUMERID,SCON_SOURCESYSTEMCODE,SCON_UNIVERSALKEY,task_id,batch_id,is_retain_for_history

In [0]:
%sql
select
      scon_mrkt_code, count(*)
from

(
      (
      select  'aus_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/AUS/aus_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='AUS' union all
      select  'hkg_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/HKG/hkg_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='HKG' union all
      select  'idn_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/IDN/idn_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='IDN' union all  
      select  'jpn_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/JPN/jpn_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='JPN' union all
      select  'kor_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/KOR/kor_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='KOR' union all
      select  'mys_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/MYS/mys_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='MYS' union all
      select  'nzl_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/NZL/nzl_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='NZL' union all
      select  'phl_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/PHL/phl_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='PHL' union all
      select  'sgp_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/SGP/sgp_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='SGP' union all
      select  'tha_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/THA/tha_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='THA' union all
      select  'twn_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/TWN/twn_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='TWN' union all
      select  'vnm_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/VNM/vnm_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='VNM'
      )

      union all

      (
      select  'aus_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/AUS/aus_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'hkg_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/HKG/hkg_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'idn_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/IDN/idn_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'jpn_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/JPN/jpn_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'kor_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/KOR/kor_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'mys_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/MYS/mys_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'nzl_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/NZL/nzl_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'phl_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/PHL/phl_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'sgp_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/SGP/sgp_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'tha_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/THA/tha_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'twn_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/TWN/twn_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'vnm_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/VNM/vnm_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true

      )
)

group by
      all
order by 
      all

In [0]:
%sql

-- normal data

select
  market_source, database_name, scon_mrkt_code, count(*), case when market_source = scon_mrkt_code then true else false end as is_sense
from

(
select 'AUS' as market_source, 'aus_elcconsumermdm' as database_name, scon_mrkt_code from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/AUS/aus_elcconsumermdm/sconsumer` 
where scon_mrkt_code = 'AUS' union all
select 'HKG' as market_source, 'hkg_elcconsumermdm' as database_name, scon_mrkt_code from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/HKG/hkg_elcconsumermdm/sconsumer` 
where scon_mrkt_code = 'HKG' union all
select 'IDN' as market_source, 'idn_elcconsumermdm' as database_name, scon_mrkt_code from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/IDN/idn_elcconsumermdm/sconsumer` 
where scon_mrkt_code = 'IDN' union all  
select 'JPN' as market_source, 'jpn_elcconsumermdm' as database_name, scon_mrkt_code from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/JPN/jpn_elcconsumermdm/sconsumer` 
where scon_mrkt_code = 'JPN' union all
select 'KOR' as market_source, 'kor_elcconsumermdm' as database_name, scon_mrkt_code from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/KOR/kor_elcconsumermdm/sconsumer` 
where scon_mrkt_code = 'KOR' union all
select 'MYS' as market_source, 'mys_elcconsumermdm' as database_name, scon_mrkt_code from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/MYS/mys_elcconsumermdm/sconsumer` 
where scon_mrkt_code = 'MYS' union all
select 'NZL' as market_source, 'nzl_elcconsumermdm' as database_name, scon_mrkt_code from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/NZL/nzl_elcconsumermdm/sconsumer` 
where scon_mrkt_code = 'NZL' union all
select 'PHL' as market_source, 'phl_elcconsumermdm' as database_name, scon_mrkt_code from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/PHL/phl_elcconsumermdm/sconsumer` 
where scon_mrkt_code = 'PHL' union all
select 'SGP' as market_source, 'sgp_elcconsumermdm' as database_name, scon_mrkt_code from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/SGP/sgp_elcconsumermdm/sconsumer` 
where scon_mrkt_code = 'SGP' union all
select 'THA' as market_source, 'tha_elcconsumermdm' as database_name, scon_mrkt_code from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/THA/tha_elcconsumermdm/sconsumer` 
where scon_mrkt_code = 'THA' union all
select 'TWN' as market_source, 'twn_elcconsumermdm' as database_name, scon_mrkt_code from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/TWN/twn_elcconsumermdm/sconsumer` 
where scon_mrkt_code = 'TWN' union all
select 'VNM' as market_source, 'vnm_elcconsumermdm' as database_name, scon_mrkt_code from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/VNM/vnm_elcconsumermdm/sconsumer` 
where scon_mrkt_code = 'VNM'
)

group by
  all
order by
  all


In [0]:
%sql

-- cross data

select
  database_name, scon_mrkt_code, count(*)
from

(
select  'aus_elcconsumermdm' as database_name, scon_mrkt_code, scon_id from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/AUS/aus_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

select  'hkg_elcconsumermdm' as database_name, scon_mrkt_code, scon_id from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/HKG/hkg_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

select  'idn_elcconsumermdm' as database_name, scon_mrkt_code, scon_id from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/IDN/idn_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

select  'jpn_elcconsumermdm' as database_name, scon_mrkt_code, scon_id from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/JPN/jpn_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

select  'kor_elcconsumermdm' as database_name, scon_mrkt_code, scon_id from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/KOR/kor_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

select  'mys_elcconsumermdm' as database_name, scon_mrkt_code, scon_id from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/MYS/mys_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

select  'nzl_elcconsumermdm' as database_name, scon_mrkt_code, scon_id from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/NZL/nzl_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

select  'phl_elcconsumermdm' as database_name, scon_mrkt_code, scon_id from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/PHL/phl_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

select  'sgp_elcconsumermdm' as database_name, scon_mrkt_code, scon_id from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/SGP/sgp_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

select  'tha_elcconsumermdm' as database_name, scon_mrkt_code, scon_id from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/THA/tha_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

select  'twn_elcconsumermdm' as database_name, scon_mrkt_code, scon_id from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/TWN/twn_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

select  'vnm_elcconsumermdm' as database_name, scon_mrkt_code, scon_id from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/VNM/vnm_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true

)

group by
  all
order by
  all


In [0]:
%sql
select
  *
from

(


select  'hkg_elcconsumermdm' as database_name, * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/HKG/hkg_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

select  'twn_elcconsumermdm' as database_name, * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/TWN/twn_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true 



)
where
    SCON_MRKT_CODE = 'AUS' and SCON_BRND_CODE = '01' and SCON_SRCS_CODE = '50aus10prlund' and SCON_CONSUMERID = '1000667294'

In [0]:
%sql

WITH kv AS (
  SELECT
    pos + 1 AS ord,
    x.Column_Name,
    x.is_null
  FROM catalog_southeastasia_mdm_silver_uat.history_data_loading.t_master_consumer
  LATERAL VIEW posexplode(array(
    named_struct('Column_Name','SCON_ID','is_null', `SCON_ID` IS NULL),
    named_struct('Column_Name','SCON_SRCC_ID','is_null', `SCON_SRCC_ID` IS NULL),
    named_struct('Column_Name','SCON_SRCC_ACTION','is_null', `SCON_SRCC_ACTION` IS NULL),
    named_struct('Column_Name','CONSUMERMDMKEY','is_null', `CONSUMERMDMKEY` IS NULL),
    named_struct('Column_Name','SCON_SRCS_CODE','is_null', `SCON_SRCS_CODE` IS NULL),
    named_struct('Column_Name','SCON_SOURCETIMESTAMP','is_null', `SCON_SOURCETIMESTAMP` IS NULL),
    named_struct('Column_Name','SCON_MRKT_CODE','is_null', `SCON_MRKT_CODE` IS NULL),
    named_struct('Column_Name','SCON_AFF_CODE','is_null', `SCON_AFF_CODE` IS NULL),
    named_struct('Column_Name','SCON_DVSN_CODE','is_null', `SCON_DVSN_CODE` IS NULL),
    named_struct('Column_Name','SCON_BRND_CODE','is_null', `SCON_BRND_CODE` IS NULL),
    named_struct('Column_Name','SCON_CONSUMERID','is_null', `SCON_CONSUMERID` IS NULL),
    named_struct('Column_Name','SCON_SALUTATION','is_null', `SCON_SALUTATION` IS NULL),
    named_struct('Column_Name','SCON_ENGLISHFIRSTNAME','is_null', `SCON_ENGLISHFIRSTNAME` IS NULL),
    named_struct('Column_Name','SCON_ENGLISHMIDDLENAME','is_null', `SCON_ENGLISHMIDDLENAME` IS NULL),
    named_struct('Column_Name','SCON_ENGLISHLASTNAME','is_null', `SCON_ENGLISHLASTNAME` IS NULL),
    named_struct('Column_Name','SCON_ENGLISHFULLNAME','is_null', `SCON_ENGLISHFULLNAME` IS NULL),
    named_struct('Column_Name','SCON_LOCALFIRSTNAME','is_null', `SCON_LOCALFIRSTNAME` IS NULL),
    named_struct('Column_Name','SCON_LOCALMIDDLENAME','is_null', `SCON_LOCALMIDDLENAME` IS NULL),
    named_struct('Column_Name','SCON_LOCALLASTNAME','is_null', `SCON_LOCALLASTNAME` IS NULL),
    named_struct('Column_Name','SCON_LOCALFULLNAME','is_null', `SCON_LOCALFULLNAME` IS NULL),
    named_struct('Column_Name','SCON_LOCALFIRSTNAME2','is_null', `SCON_LOCALFIRSTNAME2` IS NULL),
    named_struct('Column_Name','SCON_LOCALMIDDLENAME2','is_null', `SCON_LOCALMIDDLENAME2` IS NULL),
    named_struct('Column_Name','SCON_LOCALLASTNAME2','is_null', `SCON_LOCALLASTNAME2` IS NULL),
    named_struct('Column_Name','SCON_LOCALFULLNAME2','is_null', `SCON_LOCALFULLNAME2` IS NULL),
    named_struct('Column_Name','SCON_GNDR_CODE','is_null', `SCON_GNDR_CODE` IS NULL),
    named_struct('Column_Name','SCON_BIRTHDAY','is_null', `SCON_BIRTHDAY` IS NULL),
    named_struct('Column_Name','SCON_BIRTHMONTH','is_null', `SCON_BIRTHMONTH` IS NULL),
    named_struct('Column_Name','SCON_BIRTHYEAR','is_null', `SCON_BIRTHYEAR` IS NULL),
    named_struct('Column_Name','SCON_IDENTITYNUM','is_null', `SCON_IDENTITYNUM` IS NULL),
    named_struct('Column_Name','SCON_PASSPORTNUM','is_null', `SCON_PASSPORTNUM` IS NULL),
    named_struct('Column_Name','SCON_SOCIALSECURITYNUM','is_null', `SCON_SOCIALSECURITYNUM` IS NULL),
    named_struct('Column_Name','SCON_CLAS_CODE','is_null', `SCON_CLAS_CODE` IS NULL),
    named_struct('Column_Name','SCON_REG_DT','is_null', `SCON_REG_DT` IS NULL),
    named_struct('Column_Name','SCON_REGISTRATION_TOCH_CODE','is_null', `SCON_REGISTRATION_TOCH_CODE` IS NULL),
    named_struct('Column_Name','SCON_REGISTRATION_PRSN_CODE','is_null', `SCON_REGISTRATION_PRSN_CODE` IS NULL),
    named_struct('Column_Name','SCON_PREFERRED_TOCH_CODE','is_null', `SCON_PREFERRED_TOCH_CODE` IS NULL),
    named_struct('Column_Name','SCON_ASSIGNED_PRSN_CODE','is_null', `SCON_ASSIGNED_PRSN_CODE` IS NULL),
    named_struct('Column_Name','SCON_WLNG_CODE','is_null', `SCON_WLNG_CODE` IS NULL),
    named_struct('Column_Name','SCON_SLNG_CODE','is_null', `SCON_SLNG_CODE` IS NULL),
    named_struct('Column_Name','SCON_CNTR_ISOALPHA3CODE','is_null', `SCON_CNTR_ISOALPHA3CODE` IS NULL),
    named_struct('Column_Name','SCON_ETHN_CODE','is_null', `SCON_ETHN_CODE` IS NULL),
    named_struct('Column_Name','SCON_SKNT_CODE','is_null', `SCON_SKNT_CODE` IS NULL),
    named_struct('Column_Name','SCON_HAIRT_CODE','is_null', `SCON_HAIRT_CODE` IS NULL),
    named_struct('Column_Name','SCON_CVLS_CODE','is_null', `SCON_CVLS_CODE` IS NULL),
    named_struct('Column_Name','SCON_COMPANY','is_null', `SCON_COMPANY` IS NULL),
    named_struct('Column_Name','SCON_DEPARTMENT','is_null', `SCON_DEPARTMENT` IS NULL),
    named_struct('Column_Name','SCON_JOBTITLE','is_null', `SCON_JOBTITLE` IS NULL),
    named_struct('Column_Name','SCON_YEARLYINCOME','is_null', `SCON_YEARLYINCOME` IS NULL),
    named_struct('Column_Name','SCON_DONOTCONTACT_FLAG','is_null', `SCON_DONOTCONTACT_FLAG` IS NULL),
    named_struct('Column_Name','SCON_CURR_CODE','is_null', `SCON_CURR_CODE` IS NULL),
    named_struct('Column_Name','SCON_AGEFROM','is_null', `SCON_AGEFROM` IS NULL),
    named_struct('Column_Name','SCON_AGETO','is_null', `SCON_AGETO` IS NULL),
    named_struct('Column_Name','SCON_NATIONALITY','is_null', `SCON_NATIONALITY` IS NULL),
    named_struct('Column_Name','SCON_CHANNEL','is_null', `SCON_CHANNEL` IS NULL),
    named_struct('Column_Name','SCON_PREFERRED_COMM_CHANNEL','is_null', `SCON_PREFERRED_COMM_CHANNEL` IS NULL),
    named_struct('Column_Name','SCON_ANNIVERSARY_DT','is_null', `SCON_ANNIVERSARY_DT` IS NULL),
    named_struct('Column_Name','SCON_COMMERCIAL_FLAG','is_null', `SCON_COMMERCIAL_FLAG` IS NULL),
    named_struct('Column_Name','SCON_EMAILRECEIPT_FLAG','is_null', `SCON_EMAILRECEIPT_FLAG` IS NULL),
    named_struct('Column_Name','SCON_PROSPECT_FLAG','is_null', `SCON_PROSPECT_FLAG` IS NULL),
    named_struct('Column_Name','SCON_ACTIVE_FLAG','is_null', `SCON_ACTIVE_FLAG` IS NULL),
    named_struct('Column_Name','SCON_HRREQUESTTIMESTAMP','is_null', `SCON_HRREQUESTTIMESTAMP` IS NULL),
    named_struct('Column_Name','SCON_STATUS','is_null', `SCON_STATUS` IS NULL),
    named_struct('Column_Name','SCON_CREATION_DT','is_null', `SCON_CREATION_DT` IS NULL),
    named_struct('Column_Name','SCON_CREATION_UID','is_null', `SCON_CREATION_UID` IS NULL),
    named_struct('Column_Name','SCON_UPDATE_DT','is_null', `SCON_UPDATE_DT` IS NULL),
    named_struct('Column_Name','SCON_UPDATE_UID','is_null', `SCON_UPDATE_UID` IS NULL),
    named_struct('Column_Name','SCON_CBR_FLAG','is_null', `SCON_CBR_FLAG` IS NULL),
    named_struct('Column_Name','SCON_FIRSTPURCHASEDATE','is_null', `SCON_FIRSTPURCHASEDATE` IS NULL),
    named_struct('Column_Name','SCON_DELETE_FLAG','is_null', `SCON_DELETE_FLAG` IS NULL),
    named_struct('Column_Name','SCON_ENGLISHNAME_QUALITY_CODE','is_null', `SCON_ENGLISHNAME_QUALITY_CODE` IS NULL),
    named_struct('Column_Name','SCON_LOCALNAME_QUALITY_CODE','is_null', `SCON_LOCALNAME_QUALITY_CODE` IS NULL),
    named_struct('Column_Name','SCON_LOCALNAME2_QUALITY_CODE','is_null', `SCON_LOCALNAME2_QUALITY_CODE` IS NULL),
    named_struct('Column_Name','SCON_UNIVERSALKEY','is_null', `SCON_UNIVERSALKEY` IS NULL),
    named_struct('Column_Name','SCON_MASTERCONSUMERID','is_null', `SCON_MASTERCONSUMERID` IS NULL),
    named_struct('Column_Name','SCON_SOURCESYSTEMCODE','is_null', `SCON_SOURCESYSTEMCODE` IS NULL),
    named_struct('Column_Name','BATCH_ID','is_null', `BATCH_ID` IS NULL),
    named_struct('Column_Name','TASK_ID','is_null', `TASK_ID` IS NULL)
  )) pe AS pos, x
)

SELECT
  Column_Name,
  SUM(CASE WHEN is_null = false THEN 1 ELSE 0 END) AS With_Value,
  SUM(CASE WHEN is_null = true  THEN 1 ELSE 0 END) AS null_Value
FROM kv
GROUP BY ord, Column_Name
ORDER BY ord;

In [0]:
%sql

WITH talend_base AS

(
      (
      select  'aus_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/AUS/aus_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='AUS' union all
      select  'hkg_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/HKG/hkg_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='HKG' union all
      select  'idn_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/IDN/idn_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='IDN' union all  
      select  'jpn_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/JPN/jpn_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='JPN' union all
      select  'kor_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/KOR/kor_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='KOR' union all
      select  'mys_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/MYS/mys_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='MYS' union all
      select  'nzl_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/NZL/nzl_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='NZL' union all
      select  'phl_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/PHL/phl_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='PHL' union all
      select  'sgp_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/SGP/sgp_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='SGP' union all
      select  'tha_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/THA/tha_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='THA' union all
      select  'twn_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/TWN/twn_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='TWN' union all
      select  'vnm_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/VNM/vnm_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='VNM'
      )

      union all

      (
      select  'aus_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/AUS/aus_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'hkg_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/HKG/hkg_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'idn_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/IDN/idn_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'jpn_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/JPN/jpn_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'kor_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/KOR/kor_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'mys_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/MYS/mys_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'nzl_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/NZL/nzl_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'phl_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/PHL/phl_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'sgp_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/SGP/sgp_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'tha_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/THA/tha_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'twn_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/TWN/twn_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'vnm_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/VNM/vnm_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true

      )
),





In [0]:
%sql

WITH talend_base AS

(
      (
      select  'aus_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/AUS/aus_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='AUS' union all
      select  'hkg_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/HKG/hkg_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='HKG' union all
      select  'idn_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/IDN/idn_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='IDN' union all  
      select  'jpn_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/JPN/jpn_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='JPN' union all
      select  'kor_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/KOR/kor_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='KOR' union all
      select  'mys_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/MYS/mys_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='MYS' union all
      select  'nzl_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/NZL/nzl_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='NZL' union all
      select  'phl_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/PHL/phl_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='PHL' union all
      select  'sgp_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/SGP/sgp_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='SGP' union all
      select  'tha_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/THA/tha_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='THA' union all
      select  'twn_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/TWN/twn_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='TWN' union all
      select  'vnm_elcconsumermdm' as database_name,  * from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/VNM/vnm_elcconsumermdm/sconsumer` 
      where scon_mrkt_code ='VNM'
      )

      union all

      (
      select  'aus_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/AUS/aus_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'hkg_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/HKG/hkg_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'idn_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/IDN/idn_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'jpn_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/JPN/jpn_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'kor_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/KOR/kor_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'mys_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/MYS/mys_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'nzl_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/NZL/nzl_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'phl_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/PHL/phl_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'sgp_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/SGP/sgp_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'tha_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/THA/tha_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'twn_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/TWN/twn_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true union

      select  'vnm_elcconsumermdm' as database_name,  ${order_cols} from delta.`/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417/VNM/vnm_elcconsumermdm/sconsumer_cross_market` where is_retain_for_history = true

      )
),

kv AS (
  SELECT
    pos + 1 AS ord,
    x.Column_Name,
    x.is_null
  FROM talend_base
  LATERAL VIEW posexplode(array(
    named_struct('Column_Name','SCON_ID','is_null', `SCON_ID` IS NULL),
    named_struct('Column_Name','SCON_SRCC_ID','is_null', `SCON_SRCC_ID` IS NULL),
    named_struct('Column_Name','SCON_SRCC_ACTION','is_null', `SCON_SRCC_ACTION` IS NULL),
    named_struct('Column_Name','CONSUMERMDMKEY','is_null', `CONSUMERMDMKEY` IS NULL),
    named_struct('Column_Name','SCON_SRCS_CODE','is_null', `SCON_SRCS_CODE` IS NULL),
    named_struct('Column_Name','SCON_SOURCETIMESTAMP','is_null', `SCON_SOURCETIMESTAMP` IS NULL),
    named_struct('Column_Name','SCON_MRKT_CODE','is_null', `SCON_MRKT_CODE` IS NULL),
    named_struct('Column_Name','SCON_AFF_CODE','is_null', `SCON_AFF_CODE` IS NULL),
    named_struct('Column_Name','SCON_DVSN_CODE','is_null', `SCON_DVSN_CODE` IS NULL),
    named_struct('Column_Name','SCON_BRND_CODE','is_null', `SCON_BRND_CODE` IS NULL),
    named_struct('Column_Name','SCON_CONSUMERID','is_null', `SCON_CONSUMERID` IS NULL),
    named_struct('Column_Name','SCON_SALUTATION','is_null', `SCON_SALUTATION` IS NULL),
    named_struct('Column_Name','SCON_ENGLISHFIRSTNAME','is_null', `SCON_ENGLISHFIRSTNAME` IS NULL),
    named_struct('Column_Name','SCON_ENGLISHMIDDLENAME','is_null', `SCON_ENGLISHMIDDLENAME` IS NULL),
    named_struct('Column_Name','SCON_ENGLISHLASTNAME','is_null', `SCON_ENGLISHLASTNAME` IS NULL),
    named_struct('Column_Name','SCON_ENGLISHFULLNAME','is_null', `SCON_ENGLISHFULLNAME` IS NULL),
    named_struct('Column_Name','SCON_LOCALFIRSTNAME','is_null', `SCON_LOCALFIRSTNAME` IS NULL),
    named_struct('Column_Name','SCON_LOCALMIDDLENAME','is_null', `SCON_LOCALMIDDLENAME` IS NULL),
    named_struct('Column_Name','SCON_LOCALLASTNAME','is_null', `SCON_LOCALLASTNAME` IS NULL),
    named_struct('Column_Name','SCON_LOCALFULLNAME','is_null', `SCON_LOCALFULLNAME` IS NULL),
    named_struct('Column_Name','SCON_LOCALFIRSTNAME2','is_null', `SCON_LOCALFIRSTNAME2` IS NULL),
    named_struct('Column_Name','SCON_LOCALMIDDLENAME2','is_null', `SCON_LOCALMIDDLENAME2` IS NULL),
    named_struct('Column_Name','SCON_LOCALLASTNAME2','is_null', `SCON_LOCALLASTNAME2` IS NULL),
    named_struct('Column_Name','SCON_LOCALFULLNAME2','is_null', `SCON_LOCALFULLNAME2` IS NULL),
    named_struct('Column_Name','SCON_GNDR_CODE','is_null', `SCON_GNDR_CODE` IS NULL),
    named_struct('Column_Name','SCON_BIRTHDAY','is_null', `SCON_BIRTHDAY` IS NULL),
    named_struct('Column_Name','SCON_BIRTHMONTH','is_null', `SCON_BIRTHMONTH` IS NULL),
    named_struct('Column_Name','SCON_BIRTHYEAR','is_null', `SCON_BIRTHYEAR` IS NULL),
    named_struct('Column_Name','SCON_IDENTITYNUM','is_null', `SCON_IDENTITYNUM` IS NULL),
    named_struct('Column_Name','SCON_PASSPORTNUM','is_null', `SCON_PASSPORTNUM` IS NULL),
    named_struct('Column_Name','SCON_SOCIALSECURITYNUM','is_null', `SCON_SOCIALSECURITYNUM` IS NULL),
    named_struct('Column_Name','SCON_CLAS_CODE','is_null', `SCON_CLAS_CODE` IS NULL),
    named_struct('Column_Name','SCON_REG_DT','is_null', `SCON_REG_DT` IS NULL),
    named_struct('Column_Name','SCON_REGISTRATION_TOCH_CODE','is_null', `SCON_REGISTRATION_TOCH_CODE` IS NULL),
    named_struct('Column_Name','SCON_REGISTRATION_PRSN_CODE','is_null', `SCON_REGISTRATION_PRSN_CODE` IS NULL),
    named_struct('Column_Name','SCON_PREFERRED_TOCH_CODE','is_null', `SCON_PREFERRED_TOCH_CODE` IS NULL),
    named_struct('Column_Name','SCON_ASSIGNED_PRSN_CODE','is_null', `SCON_ASSIGNED_PRSN_CODE` IS NULL),
    named_struct('Column_Name','SCON_WLNG_CODE','is_null', `SCON_WLNG_CODE` IS NULL),
    named_struct('Column_Name','SCON_SLNG_CODE','is_null', `SCON_SLNG_CODE` IS NULL),
    named_struct('Column_Name','SCON_CNTR_ISOALPHA3CODE','is_null', `SCON_CNTR_ISOALPHA3CODE` IS NULL),
    named_struct('Column_Name','SCON_ETHN_CODE','is_null', `SCON_ETHN_CODE` IS NULL),
    named_struct('Column_Name','SCON_SKNT_CODE','is_null', `SCON_SKNT_CODE` IS NULL),
    named_struct('Column_Name','SCON_HAIRT_CODE','is_null', `SCON_HAIRT_CODE` IS NULL),
    named_struct('Column_Name','SCON_CVLS_CODE','is_null', `SCON_CVLS_CODE` IS NULL),
    named_struct('Column_Name','SCON_COMPANY','is_null', `SCON_COMPANY` IS NULL),
    named_struct('Column_Name','SCON_DEPARTMENT','is_null', `SCON_DEPARTMENT` IS NULL),
    named_struct('Column_Name','SCON_JOBTITLE','is_null', `SCON_JOBTITLE` IS NULL),
    named_struct('Column_Name','SCON_YEARLYINCOME','is_null', `SCON_YEARLYINCOME` IS NULL),
    named_struct('Column_Name','SCON_DONOTCONTACT_FLAG','is_null', `SCON_DONOTCONTACT_FLAG` IS NULL),
    named_struct('Column_Name','SCON_CURR_CODE','is_null', `SCON_CURR_CODE` IS NULL),
    named_struct('Column_Name','SCON_AGEFROM','is_null', `SCON_AGEFROM` IS NULL),
    named_struct('Column_Name','SCON_AGETO','is_null', `SCON_AGETO` IS NULL),
    named_struct('Column_Name','SCON_NATIONALITY','is_null', `SCON_NATIONALITY` IS NULL),
    named_struct('Column_Name','SCON_CHANNEL','is_null', `SCON_CHANNEL` IS NULL),
    named_struct('Column_Name','SCON_PREFERRED_COMM_CHANNEL','is_null', `SCON_PREFERRED_COMM_CHANNEL` IS NULL),
    named_struct('Column_Name','SCON_ANNIVERSARY_DT','is_null', `SCON_ANNIVERSARY_DT` IS NULL),
    named_struct('Column_Name','SCON_COMMERCIAL_FLAG','is_null', `SCON_COMMERCIAL_FLAG` IS NULL),
    named_struct('Column_Name','SCON_EMAILRECEIPT_FLAG','is_null', `SCON_EMAILRECEIPT_FLAG` IS NULL),
    named_struct('Column_Name','SCON_PROSPECT_FLAG','is_null', `SCON_PROSPECT_FLAG` IS NULL),
    named_struct('Column_Name','SCON_ACTIVE_FLAG','is_null', `SCON_ACTIVE_FLAG` IS NULL),
    named_struct('Column_Name','SCON_HRREQUESTTIMESTAMP','is_null', `SCON_HRREQUESTTIMESTAMP` IS NULL),
    named_struct('Column_Name','SCON_STATUS','is_null', `SCON_STATUS` IS NULL),
    named_struct('Column_Name','SCON_CREATION_DT','is_null', `SCON_CREATION_DT` IS NULL),
    named_struct('Column_Name','SCON_CREATION_UID','is_null', `SCON_CREATION_UID` IS NULL),
    named_struct('Column_Name','SCON_UPDATE_DT','is_null', `SCON_UPDATE_DT` IS NULL),
    named_struct('Column_Name','SCON_UPDATE_UID','is_null', `SCON_UPDATE_UID` IS NULL),
    named_struct('Column_Name','SCON_CBR_FLAG','is_null', `SCON_CBR_FLAG` IS NULL),
    named_struct('Column_Name','SCON_FIRSTPURCHASEDATE','is_null', `SCON_FIRSTPURCHASEDATE` IS NULL),
    named_struct('Column_Name','SCON_DELETE_FLAG','is_null', `SCON_DELETE_FLAG` IS NULL),
    named_struct('Column_Name','SCON_ENGLISHNAME_QUALITY_CODE','is_null', `SCON_ENGLISHNAME_QUALITY_CODE` IS NULL),
    named_struct('Column_Name','SCON_LOCALNAME_QUALITY_CODE','is_null', `SCON_LOCALNAME_QUALITY_CODE` IS NULL),
    named_struct('Column_Name','SCON_LOCALNAME2_QUALITY_CODE','is_null', `SCON_LOCALNAME2_QUALITY_CODE` IS NULL)
    -- ,
    -- named_struct('Column_Name','SCON_UNIVERSALKEY','is_null', `SCON_UNIVERSALKEY` IS NULL),
    -- named_struct('Column_Name','SCON_MASTERCONSUMERID','is_null', `SCON_MASTERCONSUMERID` IS NULL),
    -- named_struct('Column_Name','SCON_SOURCESYSTEMCODE','is_null', `SCON_SOURCESYSTEMCODE` IS NULL),
    -- named_struct('Column_Name','BATCH_ID','is_null', `BATCH_ID` IS NULL),
    -- named_struct('Column_Name','TASK_ID','is_null', `TASK_ID` IS NULL)
  )) pe AS pos, x
)
SELECT
  Column_Name,
  SUM(CASE WHEN is_null = false THEN 1 ELSE 0 END) AS With_Value,
  SUM(CASE WHEN is_null = true  THEN 1 ELSE 0 END) AS null_Value
FROM kv
GROUP BY ord, Column_Name
ORDER BY ord;




In [0]:
talend_snapshot_path = "/Volumes/catalog_southeastasia_mdm_share_uat/share_mdm_config/mdm_config_files/MDM_Data/Uat_Consumer_20260417"

order_cols = "scon_id,scon_srcc_id,scon_srcc_action,consumermdmkey,scon_srcs_code,scon_sourcetimestamp,scon_mrkt_code,scon_aff_code,scon_dvsn_code,scon_brnd_code,scon_consumerid,scon_salutation,scon_englishfirstname,scon_englishmiddlename,scon_englishlastname,scon_englishfullname,scon_localfirstname,scon_localmiddlename,scon_locallastname,scon_localfullname,scon_localfirstname2,scon_localmiddlename2,scon_locallastname2,scon_localfullname2,scon_gndr_code,scon_birthday,scon_birthmonth,scon_birthyear,scon_identitynum,scon_passportnum,scon_socialsecuritynum,scon_clas_code,scon_reg_dt,scon_registration_toch_code,scon_registration_prsn_code,scon_preferred_toch_code,scon_assigned_prsn_code,scon_wlng_code,scon_slng_code,scon_cntr_isoalpha3code,scon_ethn_code,scon_sknt_code,scon_hairt_code,scon_cvls_code,scon_company,scon_department,scon_jobtitle,scon_yearlyincome,scon_donotcontact_flag,scon_curr_code,scon_agefrom,scon_ageto,scon_nationality,scon_channel,scon_preferred_comm_channel,scon_anniversary_dt,scon_commercial_flag,scon_emailreceipt_flag,scon_prospect_flag,scon_active_flag,scon_hrrequesttimestamp,scon_status,scon_creation_dt,scon_creation_uid,scon_update_dt,scon_update_uid,scon_cbr_flag,scon_firstpurchasedate,scon_delete_flag,scon_englishname_quality_code,scon_localname_quality_code,scon_localname2_quality_code,is_retain_for_history"
# SCON_MASTERCONSUMERID,SCON_SOURCESYSTEMCODE,SCON_UNIVERSALKEY,task_id,batch_id,is_retain_for_history

all_talend_master_consumer_sql = f"""
(
      select 'AUS' as market_source, 'aus_elcconsumermdm' as database_name, scon_mrkt_code from delta.`{talend_snapshot_path}/AUS/aus_elcconsumermdm/sconsumer` 
      union all
      select 'HKG' as market_source, 'hkg_elcconsumermdm' as database_name, scon_mrkt_code from delta.`{talend_snapshot_path}/HKG/hkg_elcconsumermdm/sconsumer` 
      union all
      select 'IDN' as market_source, 'idn_elcconsumermdm' as database_name, scon_mrkt_code from delta.`{talend_snapshot_path}/IDN/idn_elcconsumermdm/sconsumer` 
      union all  
      select 'JPN' as market_source, 'jpn_elcconsumermdm' as database_name, scon_mrkt_code from delta.`{talend_snapshot_path}/JPN/jpn_elcconsumermdm/sconsumer` 
      union all
      select 'KOR' as market_source, 'kor_elcconsumermdm' as database_name, scon_mrkt_code from delta.`{talend_snapshot_path}/KOR/kor_elcconsumermdm/sconsumer` 
      union all
      select 'MYS' as market_source, 'mys_elcconsumermdm' as database_name, scon_mrkt_code from delta.`{talend_snapshot_path}/MYS/mys_elcconsumermdm/sconsumer` 
      union all
      select 'NZL' as market_source, 'nzl_elcconsumermdm' as database_name, scon_mrkt_code from delta.`{talend_snapshot_path}/NZL/nzl_elcconsumermdm/sconsumer` 
      union all
      select 'PHL' as market_source, 'phl_elcconsumermdm' as database_name, scon_mrkt_code from delta.`{talend_snapshot_path}/PHL/phl_elcconsumermdm/sconsumer` 
      union all
      select 'SGP' as market_source, 'sgp_elcconsumermdm' as database_name, scon_mrkt_code from delta.`{talend_snapshot_path}/SGP/sgp_elcconsumermdm/sconsumer` 
      union all
      select 'THA' as market_source, 'tha_elcconsumermdm' as database_name, scon_mrkt_code from delta.`{talend_snapshot_path}/THA/tha_elcconsumermdm/sconsumer` 
      union all
      select 'TWN' as market_source, 'twn_elcconsumermdm' as database_name, scon_mrkt_code from delta.`{talend_snapshot_path}/TWN/twn_elcconsumermdm/sconsumer` 
      union all
      select 'VNM' as market_source, 'vnm_elcconsumermdm' as database_name, scon_mrkt_code from delta.`{talend_snapshot_path}/VNM/vnm_elcconsumermdm/sconsumer` 
)
"""

corss_talend_master_consumer_sql = f"""
      (
      select  'aus_elcconsumermdm' as database_name,  {order_cols} from delta.`{talend_snapshot_path}/AUS/aus_elcconsumermdm/sconsumer_cross_market`  union

      select  'hkg_elcconsumermdm' as database_name,  {order_cols} from delta.`{talend_snapshot_path}/HKG/hkg_elcconsumermdm/sconsumer_cross_market`  union

      select  'idn_elcconsumermdm' as database_name,  {order_cols} from delta.`{talend_snapshot_path}/IDN/idn_elcconsumermdm/sconsumer_cross_market`  union

      select  'jpn_elcconsumermdm' as database_name,  {order_cols} from delta.`{talend_snapshot_path}/JPN/jpn_elcconsumermdm/sconsumer_cross_market`  union

      select  'kor_elcconsumermdm' as database_name,  {order_cols} from delta.`{talend_snapshot_path}/KOR/kor_elcconsumermdm/sconsumer_cross_market`  union

      select  'mys_elcconsumermdm' as database_name,  {order_cols} from delta.`{talend_snapshot_path}/MYS/mys_elcconsumermdm/sconsumer_cross_market`  union

      select  'nzl_elcconsumermdm' as database_name,  {order_cols} from delta.`{talend_snapshot_path}/NZL/nzl_elcconsumermdm/sconsumer_cross_market`  union

      select  'phl_elcconsumermdm' as database_name,  {order_cols} from delta.`{talend_snapshot_path}/PHL/phl_elcconsumermdm/sconsumer_cross_market`  union

      select  'sgp_elcconsumermdm' as database_name,  {order_cols} from delta.`{talend_snapshot_path}/SGP/sgp_elcconsumermdm/sconsumer_cross_market`  union

      select  'tha_elcconsumermdm' as database_name,  {order_cols} from delta.`{talend_snapshot_path}/THA/tha_elcconsumermdm/sconsumer_cross_market`  union

      select  'twn_elcconsumermdm' as database_name,  {order_cols} from delta.`{talend_snapshot_path}/TWN/twn_elcconsumermdm/sconsumer_cross_market`  union

      select  'vnm_elcconsumermdm' as database_name,  {order_cols} from delta.`{talend_snapshot_path}/VNM/vnm_elcconsumermdm/sconsumer_cross_market` 
      )
"""

spark.sql(corss_talend_master_consumer_sql).createOrReplaceGlobalTempView("corss_talend_master_consumer_tab")

In [0]:
%sql

WITH kv AS (
  SELECT
    pos + 1 AS ord,
    x.Column_Name,
    x.is_null
  FROM catalog_southeastasia_mdm_silver_uat.history_data_loading.t_master_emedia
  LATERAL VIEW posexplode(array(
    named_struct('Column_Name','SCME_ID','is_null', `SCME_ID` IS NULL),
    named_struct('Column_Name','SCME_SCON_ID','is_null', `SCME_SCON_ID` IS NULL),
    named_struct('Column_Name','SCME_MRKT_CODE','is_null', `SCME_MRKT_CODE` IS NULL),
    named_struct('Column_Name','SCME_EMDT_CODE','is_null', `SCME_EMDT_CODE` IS NULL),
    named_struct('Column_Name','SCME_SOURCETIMESTAMP','is_null', `SCME_SOURCETIMESTAMP` IS NULL),
    named_struct('Column_Name','SCME_ADDRESS','is_null', `SCME_ADDRESS` IS NULL),
    named_struct('Column_Name','SCME_VALIDITYCODE','is_null', `SCME_VALIDITYCODE` IS NULL),
    named_struct('Column_Name','SCME_PRIMARY_FLAG','is_null', `SCME_PRIMARY_FLAG` IS NULL),
    named_struct('Column_Name','SCME_APPID','is_null', `SCME_APPID` IS NULL),
    named_struct('Column_Name','SCME_CONTACTOPTINFLAG','is_null', `SCME_CONTACTOPTINFLAG` IS NULL),
    named_struct('Column_Name','SCME_REFERENCEEMEDIATYPECODE','is_null', `SCME_REFERENCEEMEDIATYPECODE` IS NULL),
    named_struct('Column_Name','SCME_REFERENCEEMEDIAADDRESS','is_null', `SCME_REFERENCEEMEDIAADDRESS` IS NULL),
    named_struct('Column_Name','SCME_QUALITY_CODE','is_null', `SCME_QUALITY_CODE` IS NULL),
    named_struct('Column_Name','SCME_QUALITY_DESC','is_null', `SCME_QUALITY_DESC` IS NULL),
    named_struct('Column_Name','SCME_CREATION_DT','is_null', `SCME_CREATION_DT` IS NULL),
    named_struct('Column_Name','SCME_CREATION_UID','is_null', `SCME_CREATION_UID` IS NULL),
    named_struct('Column_Name','SCME_UPDATE_DT','is_null', `SCME_UPDATE_DT` IS NULL),
    named_struct('Column_Name','SCME_UPDATE_UID','is_null', `SCME_UPDATE_UID` IS NULL),
    named_struct('Column_Name','SCME_UPDATE_FLAG','is_null', `SCME_UPDATE_FLAG` IS NULL),
    named_struct('Column_Name','BATCH_ID','is_null', `BATCH_ID` IS NULL),
    named_struct('Column_Name','TASK_ID','is_null', `TASK_ID` IS NULL)
  )) pe AS pos, x
)
SELECT
  Column_Name,
  SUM(CASE WHEN is_null = false THEN 1 ELSE 0 END) AS With_Value,
  SUM(CASE WHEN is_null = true  THEN 1 ELSE 0 END) AS null_Value
FROM kv
GROUP BY ord, Column_Name
ORDER BY ord;

In [0]:
all_talend_emedia_consumer_sql = f"""
(
select  'aus_elcconsumermdm' as database_name, * from delta.`{talend_snapshot_path}/AUS/aus_elcconsumermdm/sconsumermedia` 
 union all
select  'hkg_elcconsumermdm' as database_name, * from delta.`{talend_snapshot_path}/HKG/hkg_elcconsumermdm/sconsumermedia` 
 union all
select  'idn_elcconsumermdm' as database_name, * from delta.`{talend_snapshot_path}/IDN/idn_elcconsumermdm/sconsumermedia` 
 union all  
select  'jpn_elcconsumermdm' as database_name, * from delta.`{talend_snapshot_path}/JPN/jpn_elcconsumermdm/sconsumermedia` 
 union all
select  'kor_elcconsumermdm' as database_name, * from delta.`{talend_snapshot_path}/KOR/kor_elcconsumermdm/sconsumermedia` 
 union all
select  'mys_elcconsumermdm' as database_name, * from delta.`{talend_snapshot_path}/MYS/mys_elcconsumermdm/sconsumermedia` 
 union all
select  'nzl_elcconsumermdm' as database_name, * from delta.`{talend_snapshot_path}/NZL/nzl_elcconsumermdm/sconsumermedia` 
 union all
select  'phl_elcconsumermdm' as database_name, * from delta.`{talend_snapshot_path}/PHL/phl_elcconsumermdm/sconsumermedia` 
 union all
select  'sgp_elcconsumermdm' as database_name, * from delta.`{talend_snapshot_path}/SGP/sgp_elcconsumermdm/sconsumermedia` 
 union all
select  'tha_elcconsumermdm' as database_name, * from delta.`{talend_snapshot_path}/THA/tha_elcconsumermdm/sconsumermedia` 
 union all
select  'twn_elcconsumermdm' as database_name, * from delta.`{talend_snapshot_path}/TWN/twn_elcconsumermdm/sconsumermedia` 
union all
select  'vnm_elcconsumermdm' as database_name, * from delta.`{talend_snapshot_path}/VNM/vnm_elcconsumermdm/sconsumermedia` 
)
"""

spark.sql(all_talend_emedia_consumer_sql).createOrReplaceGlobalTempView("all_talend_emedia_consumer_tab")

In [0]:
%sql

WITH talend_base AS
(select
  base_tab.*
from  
  global_temp.all_talend_emedia_consumer_tab as base_tab
left join
  global_temp.corss_talend_master_consumer_tab as cross_tab
on
  base_tab.database_name = cross_tab.database_name and
  base_tab.scme_scon_id = cross_tab.scon_id 
where
  coalesce(cross_tab.is_retain_for_history, true) !=  false 
),

kv AS (
  SELECT
    pos + 1 AS ord,
    x.Column_Name,
    x.is_null
  FROM catalog_southeastasia_mdm_silver_uat.history_data_loading.t_master_emedia
  LATERAL VIEW posexplode(array(
    named_struct('Column_Name','SCME_ID','is_null', `SCME_ID` IS NULL),
    named_struct('Column_Name','SCME_SCON_ID','is_null', `SCME_SCON_ID` IS NULL),
    -- named_struct('Column_Name','SCME_MRKT_CODE','is_null', `SCME_MRKT_CODE` IS NULL),
    named_struct('Column_Name','SCME_EMDT_CODE','is_null', `SCME_EMDT_CODE` IS NULL),
    named_struct('Column_Name','SCME_SOURCETIMESTAMP','is_null', `SCME_SOURCETIMESTAMP` IS NULL),
    named_struct('Column_Name','SCME_ADDRESS','is_null', `SCME_ADDRESS` IS NULL),
    named_struct('Column_Name','SCME_VALIDITYCODE','is_null', `SCME_VALIDITYCODE` IS NULL),
    named_struct('Column_Name','SCME_PRIMARY_FLAG','is_null', `SCME_PRIMARY_FLAG` IS NULL),
    named_struct('Column_Name','SCME_APPID','is_null', `SCME_APPID` IS NULL),
    named_struct('Column_Name','SCME_CONTACTOPTINFLAG','is_null', `SCME_CONTACTOPTINFLAG` IS NULL),
    named_struct('Column_Name','SCME_REFERENCEEMEDIATYPECODE','is_null', `SCME_REFERENCEEMEDIATYPECODE` IS NULL),
    named_struct('Column_Name','SCME_REFERENCEEMEDIAADDRESS','is_null', `SCME_REFERENCEEMEDIAADDRESS` IS NULL),
    named_struct('Column_Name','SCME_QUALITY_CODE','is_null', `SCME_QUALITY_CODE` IS NULL),
    named_struct('Column_Name','SCME_QUALITY_DESC','is_null', `SCME_QUALITY_DESC` IS NULL),
    named_struct('Column_Name','SCME_CREATION_DT','is_null', `SCME_CREATION_DT` IS NULL),
    named_struct('Column_Name','SCME_CREATION_UID','is_null', `SCME_CREATION_UID` IS NULL),
    named_struct('Column_Name','SCME_UPDATE_DT','is_null', `SCME_UPDATE_DT` IS NULL),
    named_struct('Column_Name','SCME_UPDATE_UID','is_null', `SCME_UPDATE_UID` IS NULL),
    named_struct('Column_Name','SCME_UPDATE_FLAG','is_null', `SCME_UPDATE_FLAG` IS NULL)
    -- named_struct('Column_Name','BATCH_ID','is_null', `BATCH_ID` IS NULL),
    -- named_struct('Column_Name','TASK_ID','is_null', `TASK_ID` IS NULL)
  )) pe AS pos, x
)
SELECT
  Column_Name,
  SUM(CASE WHEN is_null = false THEN 1 ELSE 0 END) AS With_Value,
  SUM(CASE WHEN is_null = true  THEN 1 ELSE 0 END) AS null_Value
FROM kv
GROUP BY ord, Column_Name
ORDER BY ord;
